# 02 · El mismo dato, dos motores SQL

Ejecuta primero el cuaderno 01. Registraremos tablas externas: el catálogo describe los archivos existentes, no vuelve a copiarlos. Esta práctica usa nombres reservados a tu identificador; no modifica `aula.ventas`.

In [ ]:
from clientes import BASE, DB, hive_sql, trino_sql
hive_sql(f"CREATE DATABASE IF NOT EXISTS {DB}")
hive_sql(f"""CREATE EXTERNAL TABLE IF NOT EXISTS {DB}.ventas_csv (
 id INT, ciudad STRING, importe DOUBLE
) ROW FORMAT DELIMITED FIELDS TERMINATED BY ","
STORED AS TEXTFILE
LOCATION 'hdfs://namenode:9000{BASE}/ventas_csv'""")
hive_sql(f"""CREATE EXTERNAL TABLE IF NOT EXISTS {DB}.ventas_json (
 id INT, ciudad STRING, importe DOUBLE
) ROW FORMAT SERDE 'org.apache.hive.hcatalog.data.JsonSerDe'
STORED AS TEXTFILE
LOCATION 'hdfs://namenode:9000{BASE}/ventas_json'""")

## SQL ejecutado por HiveServer2
Escribe SQL dentro de la cadena. No escribas comandos Docker aquí.

In [ ]:
consulta = "SELECT ciudad, SUM(importe) AS total FROM {tabla} GROUP BY ciudad ORDER BY ciudad"
resultado_hive = hive_sql(consulta.format(tabla=f"{DB}.ventas_csv"))
display(resultado_hive)
assert resultado_hive.iloc[:, 1].tolist() == [150.0, 200.0]
display(hive_sql(consulta.format(tabla=f"{DB}.ventas_json")))

## La misma tabla consultada por Trino
Hive usa `base.tabla`; Trino usa `catalogo.base.tabla`. No creamos otra tabla: ambos motores comparten el metastore.

In [ ]:
resultado_trino = trino_sql(consulta.format(tabla=f"hive.{DB}.ventas_csv"))
display(resultado_trino)
assert resultado_trino.values.tolist() == resultado_hive.values.tolist()
resultado_json = trino_sql(consulta.format(tabla=f"hive.{DB}.ventas_json"))
display(resultado_json)
assert resultado_json.values.tolist() == resultado_trino.values.tolist()

**Ejercicio:** cambia la consulta para obtener la media y el número de ventas por ciudad. Ejecuta la variante con ambos motores.

Si aparece `Table not found`, comprueba el identificador y que ejecutaste las celdas de creación. Una tabla externa apunta a una carpeta: todos los archivos de datos de esa carpeta deben tener el formato declarado.